# Rainier: DAS versus seismic stations

This notebook loads the supplied CSV files and recalculates the counts, temporal matches and figures.

**Main comparison:** reviewed DAS earthquakes versus the five nearby stations (KAUT, LON, MILK, PARA and RER), using **GaMMA** for ML association. The full seismic network is reported separately for context. The 266 five-station events are accepted as valid for this comparison, as agreed; no further manual review is required.

Two questions are kept separate:

1. **TM versus ML within a sensor system:** what is shared, TM-only, or ML-only?
2. **DAS versus stations within ML:** how much do the two sensor systems overlap?

By default, matches are one-to-one within **±7 seconds**. TM stores the time of the maximum correlation, while ML stores an association origin time. The template starts approximately one second after origin, but this notebook applies **no fixed shift** because the peak-index convention and DAS clock drift have not been calibrated. These are temporal counterparts, not independently verified event identities.

Run the cells from top to bottom. Change paths, the tolerance, the TM-DAS CC filter or the time window in the configuration cell, then run all cells again. Figures and tables are exported to `results/`.


## 1. Configuration

The eight input CSVs are local research data. Use `prepare_inputs.py` to place them under `text-file/ml-tm-das-stations/inputs/` before running. The packages required for calculations are `numpy`, `scipy` and `matplotlib`; Jupyter/IPython displays the tables and figures. No pandas dependency is needed.

If packages are missing, install them in your notebook environment with `%pip install numpy scipy matplotlib` and restart the kernel before running all cells.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import csv, hashlib, html, json, math, os, platform, tempfile
import numpy as np
from scipy.optimize import linear_sum_assignment
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import maximum_bipartite_matching

os.environ.setdefault('MPLCONFIGDIR', str(Path(tempfile.gettempdir()) / 'rainier_notebook_mpl'))
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from IPython.display import display, HTML, Image, Markdown

# Locate the repository from either its root or this notebook's directory.
search_roots = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next((p for p in search_roots
                  if (p / 'text-file/ml-tm-das-stations/inputs/gamma_das_candidates.csv').is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError('Run from the repository root or notebooks/ml-tm-das-stations; keep the shared text-file inputs in this checkout.')
BASE_DIR = REPO_ROOT / 'notebooks/ml-tm-das-stations'
INPUT_DIR = REPO_ROOT / 'text-file/ml-tm-das-stations/inputs'
OUTPUT_DIR = BASE_DIR / 'results/comprehensive'
FIGURE_DIR = OUTPUT_DIR / 'figures'
MATCH_DIR = OUTPUT_DIR / 'matched_pairs'

TOLERANCE_SECONDS = 7.0
SENSITIVITY_TOLERANCES = [1, 2, 3, 5, 7, 10, 15, 20, 30]
START_UTC = '2023-08-23T21:00:00Z'       # inclusive
END_UTC = '2023-11-22T00:00:00Z'         # exclusive
TM_DAS_CC_MIN = 0.07                     # strict CC > threshold
INCLUDE_UNCERTAIN_DAS_LINKS = True       # reproduces the agreed 478-group selection
EXPORT_FILES = True

# ComCat reference uses the association's horizontal bounds, not a 25-km circle.
COMCAT_LATITUDE = (45.0, 48.0)
COMCAT_LONGITUDE = (-123.0, -120.0)
COMCAT_EVENT_TYPES = ('earthquake',)       # excludes identified explosions/quarry blasts
COMCAT_DEPTH_KM = None                    # e.g. (0.0, 40.0) to impose the nominal depth range
LOCAL_REFERENCE_RADIUS_KM = 25.0          # main local benchmark; regional box is separate
LOCAL_REFERENCE_CENTER = (46.879967, -121.726906)

FILES = {
    'DAS candidates': INPUT_DIR / 'gamma_das_candidates.csv',
    'Five stations': INPUT_DIR / 'gamma_five_station_events.csv',
    'Full network': INPUT_DIR / 'gamma_all_station_events.csv',
    'TM-DAS': INPUT_DIR / 'tm_das_detections.csv',
    'TM-stations': INPUT_DIR / 'tm_station_detections.csv',
    'Original confirmed groups': INPUT_DIR / 'gamma_original_confirmed_groups.csv',
    'Review-to-new-run mapping': INPUT_DIR / 'gamma_confirmed_group_to_new_run.csv',
    'ComCat regional source': INPUT_DIR / 'comcat_regional_raw.csv',
}
assert TOLERANCE_SECONDS > 0 and 0 <= TM_DAS_CC_MIN <= 1
for folder in [OUTPUT_DIR, FIGURE_DIR, MATCH_DIR]:
    folder.mkdir(parents=True, exist_ok=True)
missing_files = [str(p) for p in FILES.values() if not p.is_file()]
if missing_files:
    raise FileNotFoundError('Missing input CSVs. Edit BASE_DIR/FILES: ' + ', '.join(missing_files))
print(f'Input folder: {INPUT_DIR}')
print(f'Matching tolerance: ±{TOLERANCE_SECONDS:g} s; TM-DAS CC > {TM_DAS_CC_MIN:g}')
print(f'UTC interval: [{START_UTC}, {END_UTC})')


## 2. Load the CSVs and show their provenance

The original files are never edited. Each row in the station event tables is one associated event record, not one pick. `gamma_das_candidates.csv` contains all 942 new-run candidates; the review mapping below selects the previously confirmed groups.


In [ ]:
def read_csv(path):
    with Path(path).open(newline='', encoding='utf-8-sig') as handle:
        reader = csv.DictReader(handle)
        if reader.fieldnames is None:
            raise ValueError(f'CSV has no header: {path}')
        return list(reader)

def show_table(rows, columns=None, max_rows=None):
    if not rows:
        display(HTML('<p>No records.</p>'))
        return
    columns = list(columns or rows[0])
    visible = rows if max_rows is None else rows[:max_rows]
    def cell(value):
        if isinstance(value, float):
            return f'{value:,.2f}'
        return html.escape(str(value))
    head = ''.join(f'<th style="padding:7px;text-align:left">{html.escape(str(k))}</th>' for k in columns)
    body = ''.join('<tr>' + ''.join(f'<td style="padding:7px;border-bottom:1px solid #dde3e9">{cell(row.get(k, ""))}</td>' for k in columns) + '</tr>' for row in visible)
    display(HTML('<table style="border-collapse:collapse;font-family:Arial;font-size:13px"><thead style="background:#eaf0f5"><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))
    if len(visible) < len(rows):
        print(f'Showing {len(visible)} of {len(rows)} records.')

def write_csv(path, rows, columns=None):
    rows = list(rows)
    columns = list(columns or (rows[0].keys() if rows else []))
    with Path(path).open('w', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=columns, extrasaction='ignore')
        writer.writeheader()
        writer.writerows(rows)

RAW = {name: read_csv(path) for name, path in FILES.items()}
provenance = [{'Input': name, 'CSV file': path.name, 'Rows': len(RAW[name]),
               'SHA256': hashlib.sha256(path.read_bytes()).hexdigest()}
              for name, path in FILES.items()]
show_table(provenance, ['Input', 'CSV file', 'Rows'])


## 3. Reproduce the reviewed DAS selection

The original review contains **483 consolidated, confirmed groups**. The supplied mapping links **478** of those groups to distinct records in the new GaMMA run; **5** remain unmatched. Of the 478 links, 138 are flagged uncertain and are retained by default, as in the comparison already discussed.

The mapping CSV is a saved result of the earlier catalog-correspondence analysis; it does not claim a new waveform review. Changing the input event catalog may require a new mapping. Matching DAS to stations below is a separate calculation with its own tolerance.


In [ ]:
def parse_utc(text):
    value = datetime.fromisoformat(str(text).replace('Z', '+00:00'))
    return value.replace(tzinfo=timezone.utc) if value.tzinfo is None else value.astimezone(timezone.utc)

start = parse_utc(START_UTC)
end = parse_utc(END_UTC)
assert start < end

groups = RAW['Original confirmed groups']
mapping = RAW['Review-to-new-run mapping']
group_ids = {r['event_id'] for r in groups}
assert len(group_ids) == len(groups), 'Duplicate historical group IDs'
assert {r['historical_event_id'] for r in mapping} == group_ids
assert len(mapping) == len(group_ids), 'Duplicate mapping rows'
assert all(r['confirm'].lower() == 'yes' for r in groups)
assert all(r['historical_confirm'].lower() == 'yes' for r in mapping)

new_by_id = {r['event_index']: r for r in RAW['DAS candidates']}
assert len(new_by_id) == len(RAW['DAS candidates']), 'Duplicate new-run event IDs'
linked = [r for r in mapping if r['mapping_status'] == 'linked']
unlinked = [r for r in mapping if r['mapping_status'] == 'unmatched']
assert len(linked) + len(unlinked) == len(groups)
assert len({r['new_event_index'] for r in linked}) == len(linked), 'A new record was reused'
assert all(r['new_event_index'] in new_by_id for r in linked)
uncertain_links = [r for r in linked if r['historical_link_uncertain'].lower() == 'true']
selected_links = [r for r in linked if INCLUDE_UNCERTAIN_DAS_LINKS or r['historical_link_uncertain'].lower() != 'true']

review_counts = [
    {'Selection step': 'All new GaMMA-DAS candidates', 'Count': len(RAW['DAS candidates'])},
    {'Selection step': 'Original confirmed/consolidated groups', 'Count': len(groups)},
    {'Selection step': 'Groups linked to new-run records', 'Count': len(linked)},
    {'Selection step': 'Groups without an assigned new-run record', 'Count': len(unlinked)},
    {'Selection step': 'Uncertain links among linked groups', 'Count': len(uncertain_links)},
    {'Selection step': 'Linked groups retained by the current setting', 'Count': len(selected_links)},
]
show_table(review_counts)
if unlinked:
    print('Historical confirmed groups without a new-run counterpart:')
    show_table(unlinked, ['historical_event_id', 'historical_representative_idx', 'match_basis'])


## 4. Build the five catalogs used in the comparison

- **ML-DAS:** one new-run representative per selected reviewed DAS group.
- **ML-five-stations:** all events in the five-station GaMMA CSV, accepted as valid by the study assumption.
- **ML-full-network:** the full GaMMA station catalog, reported separately for context.
- **TM-DAS:** the supplied DAS detections after the agreed `CC > 0.07` selection.
- **TM-stations:** all supplied station-TM detections. Its station configuration is not established by the CSV, so it is **not relabeled as five-station TM**.

The calendar window is the common interval used in the written comparison. It is not an uptime correction. A missing temporal match does not prove a new or distinct physical earthquake.


In [ ]:
def ml_record(row, link=None):
    stamp = parse_utc(row['time'])
    return {'id': str(row['event_index']), 'time_utc': stamp.isoformat(),
            'time': stamp, 'seconds': stamp.timestamp(),
            'latitude': float(row['latitude']), 'longitude': float(row['longitude']),
            'depth_km': float(row['depth_km']), 'picks': int(row['num_picks']),
            'historical_group_id': link['historical_event_id'] if link else '',
            'historical_link_uncertain': bool(link and link['historical_link_uncertain'].lower() == 'true')}

def tm_records(rows, time_column, cc_column):
    result = []
    for row_number, row in enumerate(rows, start=2):
        stamp = parse_utc(row[time_column])
        result.append({'id': str(row_number), 'time_utc': stamp.isoformat(),
                       'time': stamp, 'seconds': stamp.timestamp(),
                       'CC': float(row[cc_column]),
                       'template_source': row.get('template_id', row.get('source_file', '')),
                       'historical_link_uncertain': False})
    return result

def within_window(records):
    records = [r for r in records if start <= r['time'] < end]
    records.sort(key=lambda r: (r['seconds'], r['id']))
    assert len({r['id'] for r in records}) == len(records)
    return records

before_window = {
    'ML-DAS': [ml_record(new_by_id[r['new_event_index']], r) for r in selected_links],
    'ML-five-stations': [ml_record(r) for r in RAW['Five stations']],
    'ML-full-network': [ml_record(r) for r in RAW['Full network']],
    'TM-DAS': [r for r in tm_records(RAW['TM-DAS'], 'UTC', 'CC') if r['CC'] > TM_DAS_CC_MIN],
    'TM-stations': tm_records(RAW['TM-stations'], 'event_time_utc', 'cc'),
}
CATALOGS = {name: within_window(rows) for name, rows in before_window.items()}
LABELS = {'ML-DAS': 'DAS: ML + GaMMA (reviewed)',
          'ML-five-stations': 'Five stations: ML + GaMMA (accepted)',
          'ML-full-network': 'Full network: ML + GaMMA (context)',
          'TM-DAS': 'DAS: template matching',
          'TM-stations': 'Stations: template matching (subset unverified)'}
totals = [{'Catalog': LABELS[name], 'Before calendar filter': len(rows),
           'Used in comparison': len(CATALOGS[name]), 'Outside interval': len(rows)-len(CATALOGS[name])}
          for name, rows in before_window.items()]
show_table(totals)
print('Full-network ML is context; the main station ML comparison uses the five-station catalog.')


## 5. One-to-one temporal matching

For each pair of catalogs, all candidate correspondences within the time tolerance are considered. The assignment maximizes the number of pairs and then minimizes their summed absolute time differences. Each record can appear at most once within that comparison. The maximum number of pairs is independently checked with a bipartite-graph matching calculation.

`Left only = left total − shared` and `Right only = right total − shared`. We also flag pairs with competing time candidates and uncertain historical DAS links. All comparisons are recalculated from the CSV-loaded records, not copied from fixed summary counts.


In [ ]:
def temporal_match(left_name, right_name, tolerance=TOLERANCE_SECONDS):
    left, right = CATALOGS[left_name], CATALOGS[right_name]
    if not left or not right:
        indices = []
        delta = np.empty((len(left), len(right)))
        eligible = np.zeros(delta.shape, dtype=bool)
    else:
        delta = np.array([r['seconds'] for r in right])[None, :] - np.array([r['seconds'] for r in left])[:, None]
        eligible = np.abs(delta) <= tolerance
        ii, jj = linear_sum_assignment(np.where(eligible, np.abs(delta)/(tolerance+1), 1e6))
        indices = [(int(i), int(j)) for i, j in zip(ii, jj) if eligible[i, j]]
        maximum = maximum_bipartite_matching(csr_matrix(eligible), perm_type='column')
        assert len(indices) == int((maximum >= 0).sum()), 'Assignment did not maximize pair count'
    used_left = {i for i, j in indices}
    used_right = {j for i, j in indices}
    assert len(used_left) == len(used_right) == len(indices)
    pairs = []
    for i, j in indices:
        a, b = left[i], right[j]
        pairs.append({'left_id': a['id'], 'right_id': b['id'],
                      'left_time_utc': a['time_utc'], 'right_time_utc': b['time_utc'],
                      'right_minus_left_s': float(delta[i, j]),
                      'left_historical_group_id': a.get('historical_group_id', ''),
                      'right_historical_group_id': b.get('historical_group_id', ''),
                      'historical_DAS_link_uncertain': bool(a.get('historical_link_uncertain') or b.get('historical_link_uncertain')),
                      'competing_time_candidates': bool(eligible[i].sum() > 1 or eligible[:, j].sum() > 1)})
    assert all(abs(p['right_minus_left_s']) <= tolerance for p in pairs)
    return {'left': left_name, 'right': right_name, 'tolerance_s': float(tolerance),
            'left_total': len(left), 'right_total': len(right), 'shared': len(pairs),
            'left_only': len(left)-len(pairs), 'right_only': len(right)-len(pairs),
            'pairs_with_competing_candidates': sum(p['competing_time_candidates'] for p in pairs),
            'pairs_with_uncertain_DAS_link': sum(p['historical_DAS_link_uncertain'] for p in pairs),
            'pairs': pairs,
            'left_unmatched': [r for i, r in enumerate(left) if i not in used_left],
            'right_unmatched': [r for j, r in enumerate(right) if j not in used_right]}

COMPARISONS = {
    'DAS: ML versus TM': ('ML-DAS', 'TM-DAS'),
    'Five-station ML versus station TM': ('ML-five-stations', 'TM-stations'),
    'Full-network ML versus station TM (context)': ('ML-full-network', 'TM-stations'),
    'ML: DAS versus five stations': ('ML-DAS', 'ML-five-stations'),
    'ML: DAS versus full network (context)': ('ML-DAS', 'ML-full-network'),
    'TM: DAS versus stations': ('TM-DAS', 'TM-stations'),
}
MATCHES = {name: temporal_match(*catalogs) for name, catalogs in COMPARISONS.items()}
summary_rows = [{'Comparison': name, 'Left total': r['left_total'], 'Shared': r['shared'],
                 'Left only': r['left_only'], 'Right total': r['right_total'], 'Right only': r['right_only'],
                 'Competing time candidates': r['pairs_with_competing_candidates']}
                for name, r in MATCHES.items()]
show_table(summary_rows)
for r in MATCHES.values():
    assert r['left_total'] == r['shared'] + r['left_only']
    assert r['right_total'] == r['shared'] + r['right_only']


## 6. Write down the TM/ML accounting

This table uses **complete selected catalogs**, without a ComCat restriction. “ML only” means no assigned TM counterpart within the selected tolerance; it does not mean “new relative to a published catalog.” The five-station ML and station-TM comparison is not a controlled five-station-versus-five-station test because the TM production station subset is unknown.


In [ ]:
accounting = []
for label, key in [('DAS', 'DAS: ML versus TM'),
                   ('Five-station ML versus existing station TM', 'Five-station ML versus station TM'),
                   ('Full-network ML (context)', 'Full-network ML versus station TM (context)')]:
    r = MATCHES[key]
    accounting.append({'System': label, 'ML total': r['left_total'], 'TM total': r['right_total'],
                       'Shared': r['shared'], 'ML only': r['left_only'], 'TM only': r['right_only']})
show_table(accounting)
for row in accounting:
    print(f"{row['System']}: ML {row['ML total']:,} = {row['Shared']:,} shared + {row['ML only']:,} ML only")
    print(f"{' '*len(row['System'])}  TM {row['TM total']:,} = {row['Shared']:,} shared + {row['TM only']:,} TM only")


## 7. Plots

The main figures keep the DAS and five-station comparison on identical count scales. The full-network total is shown separately. Every plotted count comes from the loaded and filtered records above.


In [ ]:
COLORS = {'DAS': '#75549B', 'stations': '#2B819C', 'shared': '#8798A9',
          'TM only': '#D79D37', 'ML only': '#347BA6'}
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 11,
                     'axes.spines.top': False, 'axes.spines.right': False,
                     'pdf.fonttype': 42, 'figure.facecolor': 'white', 'savefig.facecolor': 'white'})
FIGURES = []

def show_figure(fig, stem):
    path = FIGURE_DIR / (stem + '.png')
    fig.savefig(path, dpi=170, bbox_inches='tight')
    if EXPORT_FILES:
        fig.savefig(FIGURE_DIR / (stem + '.pdf'), bbox_inches='tight')
    display(Image(filename=str(path)))
    FIGURES.append({'name': stem, 'png': str(path)})
    plt.close(fig)

fig, axes = plt.subplots(1, 2, figsize=(13.8, 5.5), sharey=True)
fig.subplots_adjust(left=.08, right=.97, top=.76, bottom=.19, wspace=.20)
maximum = max(len(CATALOGS[k]) for k in ['ML-DAS', 'ML-five-stations', 'TM-DAS', 'TM-stations'])
for ax, title, keys, color in [(axes[0], 'DAS', ['TM-DAS', 'ML-DAS'], COLORS['DAS']),
                               (axes[1], 'Five nearby stations: ML', ['TM-stations', 'ML-five-stations'], COLORS['stations'])]:
    values = [len(CATALOGS[k]) for k in keys]
    ax.bar(['TM', 'ML + GaMMA'], values, color=[color, color], alpha=.85, width=.55)
    for x, n in enumerate(values):
        ax.text(x, n + maximum*.025, f'{n:,}', ha='center', fontweight='bold', fontsize=14)
    ax.set(title=title, ylabel='Records used in the comparison', ylim=(0, maximum*1.2))
    ax.grid(axis='y', alpha=.2); ax.set_axisbelow(True)
fig.suptitle('Catalog totals before temporal matching', x=.06, ha='left', fontsize=18, fontweight='bold')
fig.text(.06,.85, f"DAS ML: reviewed groups. Five-station ML: accepted events. Full-network ML (context): {len(CATALOGS['ML-full-network']):,}.", fontsize=10)
fig.text(.06,.04, 'Station TM uses the supplied catalog; its production station subset is unverified.', fontsize=10, color='#536477')
show_figure(fig, '01_catalog_totals')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14.4, 6.2), sharex=True)
fig.subplots_adjust(left=.08, right=.97, top=.70, bottom=.23, wspace=.20)
maximum = max(MATCHES[k]['left_total'] for k in ['DAS: ML versus TM', 'Five-station ML versus station TM'])
for ax, title, key in zip(axes, ['DAS', 'Five-station ML versus existing station TM'],
                         ['DAS: ML versus TM', 'Five-station ML versus station TM']):
    r = MATCHES[key]; shared = r['shared']
    ax.barh(1, shared, color=COLORS['shared'], height=.55)
    ax.barh(1, r['right_only'], left=shared, color=COLORS['TM only'], height=.55)
    ax.barh(0, shared, color=COLORS['shared'], height=.55)
    ax.barh(0, r['left_only'], left=shared, color=COLORS['ML only'], height=.55)
    for y, n in [(1, r['right_total']), (0, r['left_total'])]:
        ax.text(n+maximum*.025, y, f'{n:,}', va='center', fontweight='bold')
    ax.set(title=title, yticks=[1,0], yticklabels=['TM','ML'], xlim=(0, maximum*1.16),
           ylim=(-.65,1.65), xlabel='Catalog records (same scale)')
    ax.text(0, 1.21, f"TM: {r['right_total']} = {shared} shared + {r['right_only']} TM only", transform=ax.transAxes, fontsize=10)
    ax.text(0, 1.11, f"ML: {r['left_total']} = {shared} shared + {r['left_only']} ML only", transform=ax.transAxes, fontsize=10)
    ax.grid(axis='x', alpha=.2); ax.set_axisbelow(True)
fig.suptitle(f'TM versus ML: where each count comes from (±{TOLERANCE_SECONDS:g} s)', x=.06, ha='left', fontsize=18, fontweight='bold')
fig.legend(handles=[Patch(color=COLORS['shared'], label='Shared'), Patch(color=COLORS['TM only'], label='TM only'),
                    Patch(color=COLORS['ML only'], label='ML only')], loc='lower center', bbox_to_anchor=(.5,.08), ncol=3, frameon=False)
fig.text(.06,.025, 'Temporal counterparts using stored times; “ML only” does not establish a new physical earthquake.', fontsize=10, color='#536477')
show_figure(fig, '02_tm_ml_accounting')


In [ ]:
r = MATCHES['ML: DAS versus five stations']
fig, ax = plt.subplots(figsize=(12, 5.2))
fig.subplots_adjust(left=.14, right=.96, top=.70, bottom=.25)
for y, total, only, color in [(1, r['left_total'], r['left_only'], COLORS['DAS']),
                             (0, r['right_total'], r['right_only'], COLORS['stations'])]:
    ax.barh(y, r['shared'], color=COLORS['shared'], height=.55)
    ax.barh(y, only, left=r['shared'], color=color, height=.55)
    ax.text(total+10, y, str(total), va='center', fontweight='bold')
maximum = max(r['left_total'], r['right_total'])
ax.set(yticks=[1,0], yticklabels=['Reviewed DAS','Five stations'], xlim=(0,maximum*1.15),
       xlabel='ML + GaMMA records (same time window)', ylim=(-.65,1.65))
ax.grid(axis='x',alpha=.2);ax.set_axisbelow(True)
fig.suptitle(f'Direct ML comparison: DAS versus five stations (±{TOLERANCE_SECONDS:g} s)', x=.06, ha='left', fontsize=17, fontweight='bold')
fig.text(.06,.81, f"DAS: {r['left_total']} = {r['shared']} shared + {r['left_only']} without a station match. Five stations: {r['right_total']} = {r['shared']} + {r['right_only']}.", fontsize=11)
fig.legend(handles=[Patch(color=COLORS['shared'],label='Shared'),Patch(color=COLORS['DAS'],label='DAS without a station match'),
                    Patch(color=COLORS['stations'],label='Stations without a DAS match')],loc='lower center',bbox_to_anchor=(.52,.09),ncol=3,frameon=False,fontsize=10)
fig.text(.06,.025,'One-to-one origin-time matches; DAS clock drift remains uncorrected.',fontsize=10,color='#536477')
show_figure(fig,'03_das_five_station_overlap')


In [ ]:
hist_specs = [
    ('DAS: ML versus TM', 'DAS: TM peak − ML origin', COLORS['DAS']),
    ('Five-station ML versus station TM', 'Station TM peak − five-station ML origin', COLORS['stations']),
    ('ML: DAS versus five stations', 'Five-station origin − DAS origin', '#347BA6'),
]
fig, axes = plt.subplots(1,3,figsize=(16,5.1),sharex=True,sharey=True)
fig.subplots_adjust(left=.06,right=.98,top=.73,bottom=.23,wspace=.17)
edge = math.ceil(TOLERANCE_SECONDS)
bins = np.arange(-edge, edge+1.0, 1.0)
for ax,(key,title,color) in zip(axes,hist_specs):
    values = [p['right_minus_left_s'] for p in MATCHES[key]['pairs']]
    counts,_ = np.histogram(values,bins)
    assert int(counts.sum()) == len(values)
    ax.stairs(counts,bins,fill=True,color=color,alpha=.78)
    ax.axvline(0,color='#263647',lw=.8)
    if values:
        median = float(np.median(values));ax.axvline(median,color='#D79D37',lw=1.7)
        ax.text(.04,.94,f'n = {len(values)}\nMedian = {median:+.2f} s',transform=ax.transAxes,va='top',fontsize=10)
    ax.set(title=title,xlim=(-edge,edge),xlabel='Signed time difference (s)',ylabel='Pairs per 1 s bin')
    ax.grid(axis='y',alpha=.2);ax.set_axisbelow(True)
fig.suptitle(f'Time-difference distributions for pairs within ±{TOLERANCE_SECONDS:g} s',x=.06,ha='left',fontsize=17,fontweight='bold')
fig.text(.06,.055,'These distributions include matched pairs only. A temporal offset is not, by itself, a clock-drift estimate.',fontsize=10,color='#536477')
show_figure(fig,'04_time_difference_distributions')


In [ ]:
SENSITIVITY = {}
keys = ['DAS: ML versus TM', 'Five-station ML versus station TM', 'ML: DAS versus five stations']
fig, ax = plt.subplots(figsize=(11.5,6.8))
fig.subplots_adjust(left=.09,right=.97,top=.80,bottom=.38)
for key,color in zip(keys,[COLORS['DAS'],COLORS['stations'],'#347BA6']):
    results = [temporal_match(*COMPARISONS[key],tolerance=float(t)) for t in SENSITIVITY_TOLERANCES]
    counts = [r['shared'] for r in results]
    assert counts == sorted(counts)
    SENSITIVITY[key] = [{'tolerance_s':float(t),'shared':n} for t,n in zip(SENSITIVITY_TOLERANCES,counts)]
    ax.plot(SENSITIVITY_TOLERANCES,counts,marker='o',color=color,label=key)
ax.axvline(TOLERANCE_SECONDS,color='#536477',ls='--',lw=1)
ax.set(xlabel='Maximum absolute time difference (s)',ylabel='One-to-one matched pairs',
       xticks=sorted(set(SENSITIVITY_TOLERANCES)),xlim=(0,max(SENSITIVITY_TOLERANCES)+1))
ax.grid(alpha=.2)
fig.suptitle('How the matching tolerance changes the overlap',x=.06,ha='left',fontsize=18,fontweight='bold')
fig.legend(loc='lower center',bbox_to_anchor=(.53,.10),ncol=1,frameon=False,fontsize=10)
fig.text(.06,.025,'More matches with a wider window do not automatically establish more correct event identities.',fontsize=10,color='#536477')
show_figure(fig,'05_tolerance_sensitivity')


In [ ]:
def month_sequence(first,last):
    result=[];year,month=first.year,first.month
    while (year,month) <= (last.year,last.month):
        result.append(f'{year:04d}-{month:02d}')
        month += 1
        if month == 13:year,month=year+1,1
    return result

months=month_sequence(start,end)
fig,axes=plt.subplots(1,2,figsize=(14,6.4),sharex=True)
fig.subplots_adjust(left=.07,right=.97,top=.77,bottom=.33,wspace=.18)
x=np.arange(len(months))
for ax,title,keys in [(axes[0],'ML + GaMMA',['ML-DAS','ML-five-stations']),
                      (axes[1],'Template matching',['TM-DAS','TM-stations'])]:
    for offset,key,color in [(-.18,keys[0],COLORS['DAS']),(.18,keys[1],COLORS['stations'])]:
        counts=Counter(r['time'].strftime('%Y-%m') for r in CATALOGS[key])
        values=[counts[m] for m in months]
        assert sum(values)==len(CATALOGS[key])
        bars=ax.bar(x+offset,values,width=.34,color=color,label=LABELS[key])
        for bar,n in zip(bars,values):
            ax.annotate(str(n),(bar.get_x()+bar.get_width()/2,n),xytext=(0,3),textcoords='offset points',ha='center',fontsize=9)
    ax.set(title=title,xticks=x,xticklabels=months,ylabel='Catalog records per UTC month')
    ax.grid(axis='y',alpha=.2);ax.set_axisbelow(True)
    ax.legend(fontsize=8,loc='upper center',bbox_to_anchor=(.5,-.16),frameon=False)
fig.suptitle('Monthly distribution of the selected detections',x=.06,ha='left',fontsize=18,fontweight='bold')
fig.text(.06,.055,'August and November are partial months. These are detection counts, not uptime-normalized rates. TM station subset remains unverified.',fontsize=10,color='#536477')
show_figure(fig,'06_monthly_counts')


### One figure with panels a), b) and c)

**a)** Total records selected for each method. **b)** Within each sensor system, ML records shared with TM and ML records without a TM counterpart. **c)** Direct DAS-versus-five-station ML overlap. The full-network ML count is context only. These are different comparisons, so the shared count in b) is not the shared count in c).

The summary is exported as PNG, PDF and editable SVG. The caption is exported as a text file.


In [ ]:
plt.rcParams['svg.fonttype'] = 'none'
das_tm = MATCHES['DAS: ML versus TM']
five_tm = MATCHES['Five-station ML versus station TM']
direct = MATCHES['ML: DAS versus five stations']
fig, axes = plt.subplots(1, 3, figsize=(18.6, 6.7))
fig.subplots_adjust(left=.055, right=.98, top=.77, bottom=.37, wspace=.33)
fig.suptitle('DAS and nearby seismic stations: template matching and ML', x=.055,
             ha='left', fontsize=20, fontweight='bold')
fig.text(.055,.87, f'DAS picks: PhaseNet-DAS. Association: GaMMA. Main ML comparison: DAS versus five nearby stations; temporal window ±{TOLERANCE_SECONDS:g} s.',
         fontsize=11, color='#536477')

# a) Totals: the starting populations, without any matching subtraction.
ax = axes[0]
x = np.arange(2)
tm_counts = [das_tm['right_total'], five_tm['right_total']]
ml_counts = [das_tm['left_total'], five_tm['left_total']]
for offset, values, color, label in [(-.18, tm_counts, COLORS['TM only'], 'TM'),
                                     (.18, ml_counts, COLORS['ML only'], 'ML + GaMMA')]:
    bars = ax.bar(x+offset, values, width=.34, color=color, label=label)
    for bar, value in zip(bars, values):
        ax.annotate(f'{value:,}', (bar.get_x()+bar.get_width()/2, value),
                    xytext=(0,4), textcoords='offset points', ha='center', fontweight='bold', fontsize=11)
max_records = max(ml_counts + tm_counts)
axis_limit = math.ceil(max_records*1.15/50)*50
ax.set(xticks=x, xticklabels=['DAS', 'Station catalogs*'], ylim=(0,axis_limit), ylabel='Selected catalog records')
ax.set_title('a) Catalog totals', loc='left', fontsize=14, fontweight='bold', pad=15)
ax.grid(axis='y', alpha=.2); ax.set_axisbelow(True)
ax.legend(loc='upper center', bbox_to_anchor=(.5,-.18), ncol=2, frameon=False, fontsize=9)

# b) ML totals split relative to TM. The two missing TM pieces are written below.
ax = axes[1]
for y, r in [(1,das_tm),(0,five_tm)]:
    ax.barh(y,r['shared'],height=.55,color=COLORS['shared'])
    ax.barh(y,r['left_only'],left=r['shared'],height=.55,color=COLORS['ML only'])
    ax.text(r['shared']/2,y,str(r['shared']),ha='center',va='center',color='white',fontweight='bold',fontsize=10)
    ax.text(r['shared']+r['left_only']/2,y,str(r['left_only']),ha='center',va='center',color='white',fontweight='bold',fontsize=10)
    ax.text(r['left_total']+axis_limit*.018,y,str(r['left_total']),va='center',fontweight='bold',fontsize=10)
ax.set(yticks=[1,0],yticklabels=['DAS ML','5-station ML'],xlim=(0,axis_limit),ylim=(-.65,1.65),xlabel='ML catalog records')
ax.set_title('b) TM–ML overlap',loc='left',fontsize=14,fontweight='bold',pad=15)
ax.grid(axis='x',alpha=.2);ax.set_axisbelow(True)
ax.legend(handles=[Patch(color=COLORS['shared'],label='Shared with TM'),Patch(color=COLORS['ML only'],label='ML without TM match')],
          loc='upper center',bbox_to_anchor=(.5,-.18),ncol=1,frameon=False,fontsize=9)

# c) Sensor overlap using ML+GaMMA on both sides, independently of TM.
ax = axes[2]
for y,total,extra,color in [(1,direct['left_total'],direct['left_only'],COLORS['DAS']),
                           (0,direct['right_total'],direct['right_only'],COLORS['stations'])]:
    ax.barh(y,direct['shared'],height=.55,color=COLORS['shared'])
    ax.barh(y,extra,left=direct['shared'],height=.55,color=color)
    ax.text(direct['shared']/2,y,str(direct['shared']),ha='center',va='center',color='white',fontweight='bold',fontsize=10)
    ax.text(direct['shared']+extra/2,y,str(extra),ha='center',va='center',color='white',fontweight='bold',fontsize=10)
    ax.text(total+axis_limit*.018,y,str(total),va='center',fontweight='bold',fontsize=10)
ax.set(yticks=[1,0],yticklabels=['DAS ML','5-station ML'],xlim=(0,axis_limit),ylim=(-.65,1.65),xlabel='ML catalog records')
ax.set_title('c) DAS–station ML overlap',loc='left',fontsize=14,fontweight='bold',pad=15)
ax.grid(axis='x',alpha=.2);ax.set_axisbelow(True)
ax.legend(handles=[Patch(color=COLORS['shared'],label='Shared'),Patch(color=COLORS['DAS'],label='DAS only'),Patch(color=COLORS['stations'],label='Stations only')],
          loc='upper center',bbox_to_anchor=(.5,-.18),ncol=3,frameon=False,fontsize=9)

lefts = [a.get_position().x0 for a in axes]
fig.text(lefts[0],.19,f"Full-network ML, context only: {len(CATALOGS['ML-full-network']):,} records.",fontsize=10,color='#536477')
fig.text(lefts[1],.17,f"TM-DAS: {das_tm['right_total']} = {das_tm['shared']} shared + {das_tm['right_only']} TM only.",fontsize=10)
fig.text(lefts[1],.135,f"TM-stations: {five_tm['right_total']} = {five_tm['shared']} shared + {five_tm['right_only']} TM only.",fontsize=10)
fig.text(.055,.075,'DAS ML groups are reviewed; all five-station ML events are accepted as valid. “Only” means no assigned temporal counterpart within the chosen window.',
         fontsize=9.5,color='#536477')
fig.text(.055,.035,'*Station ML uses the five nearby stations; the TM station subset is unverified. TM peak times and ML origin times are unconverted; DAS clock drift is uncorrected.',
         fontsize=9.5,color='#536477')

summary_caption = (
    f'DAS and seismic-station comparison using GaMMA for ML association and template matching; DAS picks come from PhaseNet-DAS. '
    f'One-to-one temporal tolerance: ±{TOLERANCE_SECONDS:g} s. '
    f'(a) Selected totals: ML-DAS {das_tm["left_total"]}, ML-five-stations {five_tm["left_total"]}, '
    f'TM-DAS {das_tm["right_total"]}, and TM-stations {five_tm["right_total"]}. '
    f'The full-network ML total ({len(CATALOGS["ML-full-network"]):,}) is shown separately for context. '
    f'(b) TM/ML correspondence: DAS has {das_tm["shared"]} shared and {das_tm["left_only"]} ML-only records; '
    f'the station catalogs have {five_tm["shared"]} shared and {five_tm["left_only"]} ML-only records. '
    f'TM-only records number {das_tm["right_only"]} for DAS and {five_tm["right_only"]} for stations. '
    f'(c) Direct ML-DAS/five-station correspondence: {direct["shared"]} shared, {direct["left_only"]} DAS-only, '
    f'and {direct["right_only"]} station-only records. '
    'DAS groups are historically reviewed and linked to the new run; the five-station events are accepted as valid by the study assumption. '
    'The station-TM production subset is unverified. TM correlation-peak times and ML origin times are used as stored, '
    'with no time-reference conversion or DAS clock-drift correction. “Only” does not establish a distinct physical earthquake.'
)
if EXPORT_FILES:
    fig.savefig(FIGURE_DIR/'07_three_panel_summary.svg',bbox_inches='tight')
    (FIGURE_DIR/'07_three_panel_summary_caption.txt').write_text(summary_caption)
show_figure(fig,'07_three_panel_summary')


## 8. ComCat reference with the same latitude–longitude bounds

The regional reference is selected by **45–48° N and 123–120° W**, using the same calendar interval as the other comparisons. It replaces the 25-km circle as the regional benchmark. The input is the existing saved 300-km-radius ComCat download, which encloses this rectangle; it is not a fresh USGS query. Its rectangular subset was checked against the saved 500-km and 1000-km downloads when this notebook was updated.

Only records labeled `earthquake` are included by default. Identified explosions and quarry blasts are counted separately. The depth filter defaults to `None`, because the requested change concerns latitude and longitude. Set `COMCAT_DEPTH_KM = (0, 40)` if the reference should also match that nominal depth range.

Regional recovery and local/template-centered recovery answer different questions. A fiber or a small station subset is not expected to detect every earthquake across the regional box. Here, a counterpart is still defined by time alone; matching locations and waveform identities have not been validated.


In [ ]:
lat_min, lat_max = COMCAT_LATITUDE
lon_min, lon_max = COMCAT_LONGITUDE
assert lat_min < lat_max and lon_min < lon_max
regional_source = RAW['ComCat regional source']

def haversine_km(lat1,lon1,lat2,lon2):
    a,b = np.deg2rad([lat1,lat2]); dlon=np.deg2rad(lon2-lon1)
    h=np.sin((b-a)/2)**2+np.cos(a)*np.cos(b)*np.sin(dlon/2)**2
    return float(6371.0*2*np.arcsin(np.sqrt(np.clip(h,0,1))))

maximum_corner_distance=max(haversine_km(46.879967,-121.726906,la,lo)
                            for la in COMCAT_LATITUDE for lo in COMCAT_LONGITUDE)
if maximum_corner_distance >= 299:
    raise ValueError('The edited rectangle may extend outside the saved 300-km source catalog. Supply a wider ComCat input before comparing.')

regional_box=[]
for row in regional_source:
    stamp=parse_utc(row['time'])
    if (start <= stamp < end and lat_min <= float(row['latitude']) <= lat_max
            and lon_min <= float(row['longitude']) <= lon_max):
        regional_box.append(row)
assert len({r['id'] for r in regional_box})==len(regional_box)
type_counts=Counter(r['type'] for r in regional_box)
regional_typed=[r for r in regional_box if COMCAT_EVENT_TYPES is None or r['type'] in COMCAT_EVENT_TYPES]
regional_selected=regional_typed
if COMCAT_DEPTH_KM is not None:
    zmin,zmax=COMCAT_DEPTH_KM
    assert zmin <= zmax
    regional_selected=[r for r in regional_typed if zmin <= float(r['depth']) <= zmax]

reference_records=[]
for row in regional_selected:
    stamp=parse_utc(row['time'])
    reference_records.append({'id':row['id'],'time_utc':stamp.isoformat(),'time':stamp,'seconds':stamp.timestamp(),
                              'latitude':float(row['latitude']),'longitude':float(row['longitude']),
                              'depth_km':float(row['depth']),'magnitude':float(row['mag']) if row['mag'] else None,
                              'event_type':row['type'],'historical_link_uncertain':False})
CATALOGS['ComCat regional']=within_window(reference_records)
reference_n=len(CATALOGS['ComCat regional'])
if reference_n == 0:
    raise ValueError('No ComCat records remain after the configured filters.')

regional_filters=[
    {'Reference selection step':'All records in the saved regional source','Records':len(regional_source)},
    {'Reference selection step':'Within configured dates and latitude–longitude box','Records':len(regional_box)},
    {'Reference selection step':'After configured event-type filter','Records':len(regional_typed)},
    {'Reference selection step':'After optional depth filter: final reference','Records':reference_n},
]
show_table(regional_filters)
show_table([{'ComCat event type':k,'Records in dates and box':v} for k,v in sorted(type_counts.items())])
print(f'Horizontal reference bounds: latitude {lat_min} to {lat_max}; longitude {lon_min} to {lon_max}.')
print('Depth filter:',COMCAT_DEPTH_KM if COMCAT_DEPTH_KM is not None else 'none')
print(f'Earthquakes in the box with depth 0–40 km: {sum(r["type"]=="earthquake" and 0<=float(r["depth"])<=40 for r in regional_box)} (diagnostic; not an extra filter).')

REFERENCE_MATCHES={name:temporal_match('ComCat regional',name) for name in before_window}
REGIONAL_SUMMARY=[{'Catalog':LABELS[name],'Catalog records':r['right_total'],'ComCat reference':reference_n,
                  'Temporal counterparts':r['shared'],'Percent of ComCat':100*r['shared']/reference_n,
                  'ComCat without counterpart':r['left_only'],'Detections without reference counterpart':r['right_only'],
                  'Pairs with competing time candidates':r['pairs_with_competing_candidates']}
                 for name,r in REFERENCE_MATCHES.items()]
show_table(REGIONAL_SUMMARY)
REGIONAL_REFERENCE_INFO={'source_csv':str(FILES['ComCat regional source']),
    'cached_download_not_refreshed':True,'latitude':COMCAT_LATITUDE,'longitude':COMCAT_LONGITUDE,
    'depth_km':COMCAT_DEPTH_KM,'event_types':COMCAT_EVENT_TYPES,'reference_count':reference_n,
    'selection_counts':regional_filters,'event_type_counts':dict(type_counts),'counterpart_summary':REGIONAL_SUMMARY}


In [ ]:
fig,axes=plt.subplots(1,2,figsize=(15.8,7.3),gridspec_kw={'width_ratios':[1,1.25]})
fig.subplots_adjust(left=.07,right=.98,top=.78,bottom=.28,wspace=.38)
ax=axes[0]
refs=CATALOGS['ComCat regional']
ax.scatter([r['longitude'] for r in refs],[r['latitude'] for r in refs],s=11,c='#567A95',alpha=.65,linewidths=0)
ax.scatter(-121.760,46.853,marker='^',s=65,c='#263647',edgecolors='white',linewidths=.7,zorder=5)
ax.annotate('Rainier',(-121.760,46.853),xytext=(7,5),textcoords='offset points',fontsize=9)
ax.set(xlim=COMCAT_LONGITUDE,ylim=COMCAT_LATITUDE,xlabel='Longitude (degrees)',ylabel='Latitude (degrees)')
ax.set_aspect(1/np.cos(np.deg2rad((lat_min+lat_max)/2)))
ax.grid(alpha=.22);ax.set_title(f'a) Regional ComCat reference: {reference_n:,}',loc='left',fontweight='bold',fontsize=13)
for spine in ax.spines.values():spine.set_visible(True)

ax=axes[1]
plot_names=['ML-DAS','ML-five-stations','ML-full-network','TM-DAS','TM-stations']
plot_labels=['DAS ML + GaMMA','5-station ML + GaMMA','Full-network ML + GaMMA','DAS TM','Station TM*']
matched_color='#32988B'; unmatched_color='#DCE3E8'
largest=max(REFERENCE_MATCHES[name]['right_total'] for name in plot_names)
count_limit=math.ceil(largest*1.14/100)*100
for i,name in enumerate(plot_names):
    r=REFERENCE_MATCHES[name];matched=r['shared'];unmatched=r['right_only'];total=r['right_total']
    assert total==matched+unmatched
    ax.barh(i,matched,color=matched_color,height=.38)
    ax.barh(i,unmatched,left=matched,color=unmatched_color,height=.38)
    ax.text(total+count_limit*.015,i,f'{total:,}',va='center',fontsize=10,fontweight='bold')
    ax.text(0,i-.32,f'{total:,} total = {matched:,} ComCat matches + {unmatched:,} without match',
            va='center',fontsize=9,color='#263647')
ax.set(yticks=np.arange(5),yticklabels=plot_labels,xlim=(0,count_limit),ylim=(4.65,-.65),xlabel='All selected catalog records (not only ComCat matches)')
ax.grid(axis='x',alpha=.2);ax.set_axisbelow(True)
ax.set_title(f'b) Catalog totals and ComCat matches (±{TOLERANCE_SECONDS:g} s)',loc='left',fontweight='bold',fontsize=13)
ax.legend(handles=[Patch(color=matched_color,label='ComCat match'),Patch(color=unmatched_color,label='No ComCat match')],
          loc='upper center',bbox_to_anchor=(.5,-.14),ncol=2,frameon=False,fontsize=9)
fig.suptitle('Full catalog totals and their overlap with regional ComCat',x=.06,ha='left',fontsize=18,fontweight='bold')
depth_text='No depth filter' if COMCAT_DEPTH_KM is None else f'Depths {COMCAT_DEPTH_KM[0]:g}–{COMCAT_DEPTH_KM[1]:g} km'
type_text='Earthquakes only' if COMCAT_EVENT_TYPES == ('earthquake',) else 'All event types' if COMCAT_EVENT_TYPES is None else 'Event types: '+', '.join(COMCAT_EVENT_TYPES)
fig.text(.06,.88,f'Latitude {lat_min:g} to {lat_max:g}°; longitude {lon_min:g} to {lon_max:g}°. {type_text}. {depth_text}.',fontsize=11,color='#536477')
fig.text(.06,.11,'Each bar shows the full method catalog: total = ComCat matches + records without a ComCat match. The ComCat reference itself has a separate total.',fontsize=10,color='#536477')
fig.text(.06,.065,'*TM station subset unverified. Counterparts are based on stored times, with uncorrected TM time references and DAS clock drift.',fontsize=10,color='#536477')
fig.text(.06,.025,'Reference source: existing saved ComCat download, filtered by latitude and longitude; identified explosions/quarry blasts are excluded by default.',fontsize=10,color='#536477')
if EXPORT_FILES:fig.savefig(FIGURE_DIR/'08_regional_comcat_reference.svg',bbox_inches='tight')
show_figure(fig,'08_regional_comcat_reference')


## 9. Audit the full-network–ComCat matching

The regional percentage is not assumed to be a detection-efficiency estimate. This section checks why reference events have no counterpart: distance from the original Rainier query center, matching tolerance, a possible global UTC-hour offset, and the horizontal separation of time-paired locations.

The station CSV has timezone-naive timestamps; they are interpreted as UTC, consistently with the original association workflow. The tests below do not shift or overwrite any event times. A reference-event position inside the association search rectangle does not establish that the Rainier station set can detect that event or that its picks pass association thresholds.


In [ ]:
audit_ref=CATALOGS['ComCat regional']
audit_ml=CATALOGS['ML-full-network']
audit_baseline=REFERENCE_MATCHES['ML-full-network']
audit_delta=np.array([r['seconds'] for r in audit_ml])[None,:]-np.array([r['seconds'] for r in audit_ref])[:,None]
audit_abs=np.abs(audit_delta)
AUDIT_CENTER=LOCAL_REFERENCE_CENTER  # original Rainier reference-query center
audit_radius=np.array([haversine_km(*AUDIT_CENTER,r['latitude'],r['longitude']) for r in audit_ref])
ref_by_id={r['id']:r for r in audit_ref};ml_by_id={r['id']:r for r in audit_ml}
matched_reference_ids={p['left_id'] for p in audit_baseline['pairs']}

def audit_assignment_count(tolerance,offset_seconds=0):
    difference=np.abs(audit_delta+offset_seconds)
    valid=difference<=tolerance
    ii,jj=linear_sum_assignment(np.where(valid,difference/(tolerance+1),1e6))
    count=sum(bool(valid[i,j]) for i,j in zip(ii,jj))
    independent=maximum_bipartite_matching(csr_matrix(valid),perm_type='column')
    assert count==int((independent>=0).sum())
    return count

MATCH_AUDIT_TOLERANCES=[1,2,5,7,10,15,20,30,60,120,300]
audit_tolerance=[{'Tolerance_s':t,'One-to-one counterparts':audit_assignment_count(t)} for t in MATCH_AUDIT_TOLERANCES]
assert audit_assignment_count(TOLERANCE_SECONDS)==audit_baseline['shared']
audit_any=int((audit_abs<=TOLERANCE_SECONDS).any(axis=1).sum())
print(f"At ±{TOLERANCE_SECONDS:g} s: {audit_baseline['shared']} assigned matches; {audit_any} ComCat events have any eligible temporal candidate.")
print('A difference between these counts would indicate competition for the same ML record.')
show_table(audit_tolerance)

distance_bins=[(0,25,'0–25 km'),(25,50,'25–50 km'),(50,100,'50–100 km'),(100,150,'100–150 km'),(150,float('inf'),'150+ km')]
audit_distance=[]
for low,high,label in distance_bins:
    rows=[r for r,dist in zip(audit_ref,audit_radius) if low<=dist<high]
    n=sum(r['id'] in matched_reference_ids for r in rows)
    audit_distance.append({'Distance from original Rainier query center':label,'ComCat events':len(rows),
                           'Temporal counterparts':n,'Without counterpart':len(rows)-n,
                           'Percent with counterpart':100*n/len(rows) if rows else 0})
assert sum(r['ComCat events'] for r in audit_distance)==len(audit_ref)
assert sum(r['Temporal counterparts'] for r in audit_distance)==audit_baseline['shared']
show_table(audit_distance)

# Local benchmark is anchored on ComCat positions. Do not also clip ML locations:
# location errors at the boundary would turn a detected event into a false miss.
local_reference_records=[r for r,dist in zip(audit_ref,audit_radius) if dist<=LOCAL_REFERENCE_RADIUS_KM]
CATALOGS['ComCat local']=local_reference_records
LOCAL_MATCHES={name:temporal_match('ComCat local',name) for name in before_window}
LOCAL_SUMMARY=[{'Catalog':LABELS[name],'Local ComCat reference':len(local_reference_records),
                'Counterparts':r['shared'],'Local ComCat without counterpart':r['left_only'],
                'Percent with counterpart':100*r['shared']/len(local_reference_records) if local_reference_records else 0}
               for name,r in LOCAL_MATCHES.items()]
display(Markdown(f'**Primary local benchmark: ComCat events within {LOCAL_REFERENCE_RADIUS_KM:g} km.** The regional-box comparison remains a separate diagnostic.'))
show_table(LOCAL_SUMMARY)
print('Reference positions define the local subset; all selected ML records remain eligible for a time match.')

# In the reverse comparison, select DAS by its estimated location and retain
# matches against the entire regional reference, including outside the circle.
local_das_records=[dict(r,distance_from_rainier_km=haversine_km(*AUDIT_CENTER,r['latitude'],r['longitude']))
                   for r in CATALOGS['ML-DAS']
                   if haversine_km(*AUDIT_CENTER,r['latitude'],r['longitude'])<=LOCAL_REFERENCE_RADIUS_KM]
regional_das_matched_ids={p['right_id'] for p in REFERENCE_MATCHES['ML-DAS']['pairs']}
local_das_extras=[r for r in local_das_records if r['id'] not in regional_das_matched_ids]
LOCAL_DAS_ACCOUNTING={'DAS_within_radius':len(local_das_records),
                      'With_regional_ComCat_counterpart':len(local_das_records)-len(local_das_extras),
                      'Without_regional_ComCat_counterpart':len(local_das_extras),
                      'Uncertain_historical_links_in_unmatched_subset':sum(r['historical_link_uncertain'] for r in local_das_extras)}
show_table([LOCAL_DAS_ACCOUNTING])

audit_hours=[{'Offset added to station ML (hours)':h,'Counterparts at current tolerance':audit_assignment_count(TOLERANCE_SECONDS,h*3600)} for h in range(-12,13)]
best_hour=max(audit_hours,key=lambda r:r['Counterparts at current tolerance'])
print('Best global hour offset:',best_hour)
print('These shifts are diagnostics only; no clock correction is applied.')

audit_spatial=[]
for pair in audit_baseline['pairs']:
    a,b=ref_by_id[pair['left_id']],ml_by_id[pair['right_id']]
    separation=haversine_km(a['latitude'],a['longitude'],b['latitude'],b['longitude'])
    audit_spatial.append({'ComCat_ID':a['id'],'ComCat_origin_UTC':a['time_utc'],
                          'ML_ID':b['id'],'ML_origin_UTC':b['time_utc'],
                          'ML_minus_ComCat_s':pair['right_minus_left_s'],
                          'Horizontal_separation_km':separation,'ComCat_magnitude':a['magnitude'],
                          'Reference_distance_from_center_km':haversine_km(*AUDIT_CENTER,a['latitude'],a['longitude'])})
spatial_values=[r['Horizontal_separation_km'] for r in audit_spatial]
signed_values=[r['ML_minus_ComCat_s'] for r in audit_spatial]
audit_statistics={
    'reference_events':len(audit_ref),'station_ML_records':len(audit_ml),
    'current_tolerance_s':TOLERANCE_SECONDS,'time_only_counterparts':audit_baseline['shared'],
    'references_with_any_temporal_candidate':audit_any,
    'median_ML_minus_ComCat_seconds':float(np.median(signed_values)),
    'median_horizontal_separation_km':float(np.median(spatial_values)),
    'p90_horizontal_separation_km':float(np.quantile(spatial_values,.90)),
    'time_pairs_separated_by_more_than_25km':sum(v>25 for v in spatial_values),
    'best_global_hour_offset':best_hour,
}
show_table([audit_statistics])
print('Time-paired events with the largest location differences (flag for review, not automatically rejected):')
show_table(sorted(audit_spatial,key=lambda r:r['Horizontal_separation_km'],reverse=True),max_rows=8)

audit_unmatched=[]
for i,row in enumerate(audit_ref):
    if row['id'] in matched_reference_ids:continue
    nearest=int(np.argmin(audit_abs[i]));candidate=audit_ml[nearest]
    audit_unmatched.append({'ComCat_ID':row['id'],'Origin_UTC':row['time_utc'],
                            'Latitude':row['latitude'],'Longitude':row['longitude'],
                            'Depth_km':row['depth_km'],'Magnitude':row['magnitude'],
                            'Distance_from_center_km':float(audit_radius[i]),
                            'Nearest_ML_ID':candidate['id'],
                            'Nearest_ML_minus_ComCat_s':float(audit_delta[i,nearest]),
                            'Nearest_time_candidate_separation_km':haversine_km(row['latitude'],row['longitude'],candidate['latitude'],candidate['longitude'])})
print('Unmatched ComCat events within the original 25-km region:')
show_table([r for r in audit_unmatched if r['Distance_from_center_km']<25])
MATCH_AUDIT={'statistics':audit_statistics,'distance_bands':audit_distance,'tolerance_sensitivity':audit_tolerance,
             'global_hour_offset_test':audit_hours,
             'interpretation':'Time-only counterparts. Spatially discrepant pairs require waveform/pick inspection. The event CSVs alone do not identify why a reference earthquake lacks station picks or an association.'}


In [ ]:
fig,axes=plt.subplots(1,3,figsize=(18.6,6.8),gridspec_kw={'width_ratios':[1,1.15,1]})
fig.subplots_adjust(left=.055,right=.98,top=.75,bottom=.29,wspace=.36)
fig.suptitle('Audit: why does the regional ComCat match percentage look low?',x=.055,ha='left',fontsize=19,fontweight='bold')
fig.text(.055,.86,f'Full Rainier station set + GaMMA: {len(audit_ml):,} records. Regional ComCat: {len(audit_ref):,}. Time-only pairing at ±{TOLERANCE_SECONDS:g} s.',fontsize=11,color='#536477')
ax=axes[0]
for matched,color,label in [(False,'#CB7771','No temporal counterpart'),(True,'#32988B','Temporal counterpart')]:
    subset=[r for r in audit_ref if (r['id'] in matched_reference_ids)==matched]
    ax.scatter([r['longitude'] for r in subset],[r['latitude'] for r in subset],s=10,c=color,alpha=.72,linewidths=0,label=label)
for radius,style in [(25,'--'),(50,':')]:
    bearing=np.linspace(0,2*np.pi,180);angular=radius/6371
    la,lo=np.deg2rad(AUDIT_CENTER)
    dest_lat=np.arcsin(np.sin(la)*np.cos(angular)+np.cos(la)*np.sin(angular)*np.cos(bearing))
    dest_lon=lo+np.arctan2(np.sin(bearing)*np.sin(angular)*np.cos(la),np.cos(angular)-np.sin(la)*np.sin(dest_lat))
    ax.plot(np.rad2deg(dest_lon),np.rad2deg(dest_lat),color='#263647',ls=style,lw=.9)
ax.scatter(AUDIT_CENTER[1],AUDIT_CENTER[0],marker='^',s=45,c='#263647',zorder=5)
ax.set(xlim=COMCAT_LONGITUDE,ylim=COMCAT_LATITUDE,xlabel='Longitude',ylabel='Latitude')
ax.set_aspect(1/np.cos(np.deg2rad(np.mean(COMCAT_LATITUDE))))
ax.set_title('a) Where the nonmatches are',loc='left',fontsize=13,fontweight='bold');ax.grid(alpha=.18)
ax.legend(loc='upper center',bbox_to_anchor=(.5,-.16),frameon=False,fontsize=8)

ax=axes[1]
aggregated=[]
for low,high,label in [(0,25,'0–25 km'),(25,50,'25–50 km'),(50,float('inf'),'>50 km')]:
    subset=[r for r,dist in zip(audit_ref,audit_radius) if low<=dist<high]
    shared=sum(r['id'] in matched_reference_ids for r in subset)
    aggregated.append((label,len(subset),shared))
for y,(label,total,shared) in enumerate(aggregated):
    fraction=100*shared/total if total else 0
    ax.barh(y,fraction,color='#32988B',height=.5)
    ax.text(2,y-.33,f'{shared}/{total} ({fraction:.1f}%)',fontsize=10,fontweight='semibold')
ax.set(yticks=range(3),yticklabels=[r[0] for r in aggregated],xlim=(0,105),ylim=(2.65,-.65),xlabel='ComCat events with a temporal counterpart (%)')
ax.set_title('b) Separate distance ranges',loc='left',fontsize=13,fontweight='bold');ax.grid(axis='x',alpha=.2);ax.set_axisbelow(True)

ax=axes[2]
limits=[r['Tolerance_s'] for r in audit_tolerance if r['Tolerance_s']<=120]
counts=[r['One-to-one counterparts'] for r in audit_tolerance if r['Tolerance_s']<=120]
ax.plot(limits,counts,'o-',c='#347BA6',ms=4)
ax.axvline(TOLERANCE_SECONDS,color='#536477',ls='--',lw=1)
ax.set(xscale='log',xticks=[1,5,7,15,30,60,120],xticklabels=['1','5','7','15','30','60','120'],ylim=(0,max(counts)*1.15),
       xlabel='Maximum origin-time difference (s)',ylabel='One-to-one counterparts')
ax.set_title('c) Is the tolerance too strict?',loc='left',fontsize=13,fontweight='bold');ax.grid(alpha=.2)
for limit in [TOLERANCE_SECONDS,60]:
    count=next(r['One-to-one counterparts'] for r in audit_tolerance if r['Tolerance_s']==limit)
    ax.annotate(f'{count} at ±{limit:g} s',(limit,count),xytext=(5,-18 if limit==TOLERANCE_SECONDS else 10),textcoords='offset points',fontsize=9)

fig.text(.055,.12,'Distance is measured from the original Rainier query center. Map rings: 25 km (dashed) and 50 km (dotted). Wider time windows can introduce chance matches.',fontsize=9.5,color='#536477')
fig.text(.055,.075,f"Median matched station-origin minus ComCat-origin difference: {audit_statistics['median_ML_minus_ComCat_seconds']:+.3f} s. Location discrepancies remain flagged in the audit tables.",fontsize=9.5,color='#536477')
fig.text(.055,.03,'This audit establishes where the mismatch occurs. Pick-level records, station availability and waveforms are needed to establish its physical or processing cause.',fontsize=9.5,color='#536477')
if EXPORT_FILES:fig.savefig(FIGURE_DIR/'09_full_network_match_audit.svg',bbox_inches='tight')
show_figure(fig,'09_full_network_match_audit')


## 10. Inspect individual pairs and export the results

The matched-pair CSVs retain both IDs, both timestamps, the signed time difference, ambiguity flags and the historical DAS group ID. This allows any number in a plot to be traced to its records. Original input CSVs are never overwritten.

The full-network comparison is included in the summary for context; it is not substituted for the five-station comparison in the main figures.


In [ ]:
example_key = 'ML: DAS versus five stations'  # change this to inspect another comparison
show_table(MATCHES[example_key]['pairs'], max_rows=12)

if EXPORT_FILES:
    write_csv(OUTPUT_DIR/'catalog_totals.csv',totals)
    write_csv(OUTPUT_DIR/'comparison_summary.csv',summary_rows)
    write_csv(OUTPUT_DIR/'tm_ml_accounting.csv',accounting)
    write_csv(OUTPUT_DIR/'input_provenance.csv',provenance)
    write_csv(OUTPUT_DIR/'regional_comcat_counterparts.csv',REGIONAL_SUMMARY)
    write_csv(OUTPUT_DIR/'full_network_audit_distance.csv',audit_distance)
    write_csv(OUTPUT_DIR/'full_network_audit_tolerance.csv',audit_tolerance)
    write_csv(OUTPUT_DIR/'full_network_audit_hour_offsets.csv',audit_hours)
    write_csv(OUTPUT_DIR/'full_network_audit_spatial_pairs.csv',audit_spatial)
    write_csv(OUTPUT_DIR/'full_network_unmatched_reference_events.csv',audit_unmatched)
    write_csv(OUTPUT_DIR/'local_comcat_counterpart_summary.csv',LOCAL_SUMMARY)
    write_csv(OUTPUT_DIR/'local_das_comcat_accounting.csv',[LOCAL_DAS_ACCOUNTING])
    write_csv(OUTPUT_DIR/'local_das_without_comcat_counterpart.csv',
              [{k:v for k,v in r.items() if k not in ['time','seconds']} for r in local_das_extras])
    write_csv(OUTPUT_DIR/'local_comcat_unmatched_full_network.csv',
              [r for r in audit_unmatched if r['Distance_from_center_km']<=LOCAL_REFERENCE_RADIUS_KM])
    (OUTPUT_DIR/'full_network_match_audit.json').write_text(json.dumps(MATCH_AUDIT,indent=2)+'\n')
    write_csv(OUTPUT_DIR/'regional_comcat_reference.csv',
              [{k:v for k,v in r.items() if k not in ['time','seconds']} for r in CATALOGS['ComCat regional']])
    for index,(name,r) in enumerate(REFERENCE_MATCHES.items(),start=1):
        write_csv(MATCH_DIR/f'regional_reference_{index:02d}_pairs.csv',r['pairs'],
                  ['left_id','right_id','left_time_utc','right_time_utc','right_minus_left_s',
                   'left_historical_group_id','right_historical_group_id',
                   'historical_DAS_link_uncertain','competing_time_candidates'])
    for index,(name,r) in enumerate(MATCHES.items(),start=1):
        write_csv(MATCH_DIR/f'{index:02d}_matched_pairs.csv',r['pairs'],
                  ['left_id','right_id','left_time_utc','right_time_utc','right_minus_left_s',
                   'left_historical_group_id','right_historical_group_id',
                   'historical_DAS_link_uncertain','competing_time_candidates'])
    summary = {'configuration':{'start_inclusive':START_UTC,'end_exclusive':END_UTC,
                               'tolerance_seconds':TOLERANCE_SECONDS,'tm_das_cc_strict_min':TM_DAS_CC_MIN,
                               'include_uncertain_DAS_links':INCLUDE_UNCERTAIN_DAS_LINKS},
               'catalog_totals':totals,'review_selection':review_counts,
               'comparisons':summary_rows,'tm_ml_accounting':accounting,
               'tolerance_sensitivity':SENSITIVITY,'input_provenance':provenance,
               'regional_comcat_reference':REGIONAL_REFERENCE_INFO,
               'full_network_match_audit':MATCH_AUDIT,
               'local_comcat_benchmark':{'radius_km':LOCAL_REFERENCE_RADIUS_KM,'center':LOCAL_REFERENCE_CENTER,
                                         'reference_events':len(local_reference_records),'summary':LOCAL_SUMMARY},
               'local_DAS_without_reference_accounting':LOCAL_DAS_ACCOUNTING,
               'runtime':{'python':platform.python_version(),'numpy':np.__version__,'matplotlib':matplotlib.__version__},
               'figures':FIGURES}
    (OUTPUT_DIR/'analysis_summary.json').write_text(json.dumps(summary,indent=2)+'\n')
    print(f'Exported figures and tables to {OUTPUT_DIR}')

display(Markdown(f"**Current main comparison:** {len(CATALOGS['ML-DAS'])} reviewed DAS groups versus {len(CATALOGS['ML-five-stations'])} accepted five-station events. Full-network ML: {len(CATALOGS['ML-full-network']):,} records in the comparison window."))


## Interpretation

- Use the **direct DAS–five-station overlap** to discuss complementary observations from the two systems.
- Use the **TM–ML accounting** to discuss how the selected methods overlap within the supplied catalogs. The station-TM production configuration is not verified as the five-station array.
- The 266 five-station events are accepted as valid by the study assumption. DAS groups were visually confirmed in the historical review, then linked to the new run; uncertain links remain visible in the inputs and pair tables.
- “Only in one catalog” means no assigned temporal counterpart within the chosen window. Time-reference differences, clock drift, source location, association changes and raw-data availability can all affect that result.
- These calculations do not measure location accuracy, absolute completeness or a validated false-positive rate. Catalogs are kept separate; the same earthquake appearing in GaMMA-DAS and station-GaMMA is not added twice to a new combined catalog.
